In [0]:
sector_lookup = spark.sql("SELECT ticker, sector from plstocks.silver_sector_lookup order by sector ASC")
all_sectors = spark.sql("SELECT DISTINCT sector from plstocks.silver_sector_lookup")
active_stock = spark.sql("SELECT DISTINCT ticker from plstocks.silver_stocks_price where active = True")
ticker_per_sector = active_stock.join(sector_lookup, active_stock.ticker == sector_lookup.ticker, how = 'inner')
ticker_per_sector = ticker_per_sector.select('plstocks.silver_stocks_price.ticker', 'sector')
display(spark.sql("SELECT * from plstocks.silver_sector_lookup order by ticker ASC"))

In [0]:

current_price = spark.sql("SELECT ticker, close AS current_price FROM plstocks.silver_stocks_price WHERE DATE = (SELECT MAX(DATE) FROM plstocks.silver_stocks_price)")
one_m_price = spark.sql("SELECT ticker, close AS 1m_price FROM plstocks.silver_stocks_price WHERE DATE = (SELECT MAX(DATE) FROM plstocks.silver_stocks_price WHERE DATE < ADD_MONTHS(CURRENT_DATE, -1))")
three_m_price = spark.sql("SELECT ticker, close AS 3m_price FROM plstocks.silver_stocks_price WHERE DATE = (SELECT MAX(DATE) FROM plstocks.silver_stocks_price WHERE DATE < ADD_MONTHS(CURRENT_DATE, -3))")
six_m_price = spark.sql("SELECT ticker, close AS 6m_price FROM plstocks.silver_stocks_price WHERE DATE = (SELECT MAX(DATE) FROM plstocks.silver_stocks_price WHERE DATE < ADD_MONTHS(CURRENT_DATE, -6))")
one_y_price = spark.sql("SELECT ticker, close AS 1y_price FROM plstocks.silver_stocks_price WHERE DATE = (SELECT MAX(DATE) FROM plstocks.silver_stocks_price WHERE DATE < ADD_MONTHS(CURRENT_DATE, -12))")
three_y_price = spark.sql("SELECT ticker, close AS 3y_price FROM plstocks.silver_stocks_price WHERE DATE = (SELECT MAX(DATE) FROM plstocks.silver_stocks_price WHERE DATE < ADD_MONTHS(CURRENT_DATE, -36))")
five_y_price = spark.sql("SELECT ticker, close AS 5y_price FROM plstocks.silver_stocks_price WHERE DATE = (SELECT MAX(DATE) FROM plstocks.silver_stocks_price WHERE DATE < ADD_MONTHS(CURRENT_DATE, -60))")
prices_timestamps = (
    current_price
    .join(one_m_price, ["ticker"], "left")
    .join(three_m_price, ["ticker"], "left")
    .join(six_m_price, ["ticker"], "left")
    .join(one_y_price, ["ticker"], "left")
    .join(three_y_price, ["ticker"], "left")
    .join(five_y_price, ["ticker"], "left")
)
display(prices_timestamps.orderBy("ticker"))

In [0]:
from pyspark.sql.functions import udf

@udf('float')
def percent_change(current_price, old_price):
    if current_price is None or old_price is None:
        return None 
    current_price = float(current_price)
    old_price = float(old_price)
    if old_price - current_price == 0:
        return 0.0
    return ((current_price - old_price) / current_price) * 100

prices_timestamps = (
    prices_timestamps
    .withColumn('1m_change', percent_change(prices_timestamps.current_price, prices_timestamps['1m_price']))
    .withColumn('3m_change', percent_change(prices_timestamps.current_price, prices_timestamps['3m_price']))
    .withColumn('6m_change', percent_change(prices_timestamps.current_price, prices_timestamps['6m_price']))
    .withColumn('1y_change', percent_change(prices_timestamps.current_price, prices_timestamps['1y_price']))
    .withColumn('3y_change', percent_change(prices_timestamps.current_price, prices_timestamps['3y_price']))
    .withColumn('5y_change', percent_change(prices_timestamps.current_price, prices_timestamps['5y_price']))
)
display(prices_timestamps.orderBy("ticker"))

In [0]:
prices_timestamps.join(sector_lookup, on='ticker', how='inner').createOrReplaceTempView("prices_timestamps_with_sectors")
sector_aggregation = spark.sql("SELECT sector, ROUND(AVG(3m_change),2) AS avg_3m_return, ROUND(AVG(6m_change),2) AS avg_6m_change, ROUND(AVG(1y_change),2) AS avg_1y_change, ROUND(AVG(3y_change),2) AS avg_3y_change, ROUND(AVG(5y_change),2) AS avg_5y_change FROM prices_timestamps_with_sectors GROUP BY sector ORDER BY sector ASC")
display(sector_aggregation)

In [0]:
spark.sql("SELECT sector, MAX(1y_change) AS max_1y_change FROM prices_timestamps_with_sectors GROUP BY sector ORDER BY sector ASC").createOrReplaceTempView("max_1y_changes")
display(spark.sql("SELECT * FROM max_1y_changes"))

In [0]:
spark.sql("SELECT mc.sector, pt.ticker FROM prices_timestamps_with_sectors pt INNER JOIN max_1y_changes mc ON pt.sector = mc.sector AND pt.1y_change = mc.max_1y_change ").createOrReplaceTempView("best_performing_stocks_per_sector")
display(spark.sql("SELECT * FROM best_performing_stocks_per_sector"))

In [0]:
sector_aggregation.createOrReplaceTempView("sector_aggregation")
sector_aggregation_with_stocks = spark.sql("SELECT s.*, b.ticker AS best_performing_stock FROM sector_aggregation s INNER JOIN best_performing_stocks_per_sector b ON s.sector = b.sector")
display(sector_aggregation_with_stocks)

In [0]:
import pyspark.sql.functions as F

current_price.createOrReplaceTempView("current_price")
div_yields_for_stocks = spark.sql("SELECT ROUND((d.dividend_per_share/c.current_price)*100,2) AS dividend_yield, d.ticker FROM plstocks.silver_dividend_history d INNER JOIN current_price c ON d.ticker = c.ticker WHERE d.dividend_year = YEAR(CURRENT_DATE)-1")
div_yields_for_stocks_with_sector = div_yields_for_stocks.join(ticker_per_sector, on=['ticker'], how='right')
avg_div_yield_per_sector = div_yields_for_stocks_with_sector.groupBy('sector').agg({'dividend_yield': 'avg'})
avg_div_yield_per_sector = avg_div_yield_per_sector.withColumn('avg(dividend_yield)', F.round('avg(dividend_yield)', 2)).withColumnRenamed('avg(dividend_yield)', 'average_dividend_yield')
display(avg_div_yield_per_sector)

In [0]:
sector_aggregation_with_stocks_and_div = sector_aggregation_with_stocks.join(avg_div_yield_per_sector, on='sector', how='left')
display(sector_aggregation_with_stocks_and_div)

In [0]:
sector_aggregation_with_stocks_and_div.write.mode("overwrite").saveAsTable("plstocks.gold_sector_performance")